In [74]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict,List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing_extensions import Annotated
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
import operator

In [75]:
load_dotenv()

True

In [76]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
) 

model = ChatHuggingFace(llm = llm)



 

In [77]:
class EssayState(TypedDict):
   essay : str
   topic : str
   language_feedback : str
   analysis_feedback : str
   clarity_feedback : str
   overall_feedback : str
   individual_score : Annotated[list['int'], operator.add]
   final_avg_score : float

In [78]:
class EvaluationSchema(BaseModel):
   feedback : str = Field(description= 'detailed feed back for the essay')
   score : int = Field(description='score out of 10', ge=0, le= 10)


In [79]:
parser = PydanticOutputParser(pydantic_object=EvaluationSchema)

In [80]:
essay_prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
Write a UPSC-style essay of 250–300 words on the topic:

"{topic}"

Guidelines:
- Clear introduction
- Multidimensional analysis
- Examples where relevant
- Short conclusion
"""
)


In [81]:
def generate_essay(state: EssayState):
    response = model.invoke(
        essay_prompt.format(topic=state["topic"])
    )


    return {
        "essay": response.content
    }

In [82]:
def evaluation_language(state : EssayState):
    prompt = PromptTemplate(
         template = "Evaluate the language quality of the followimng essay and provide a feedback and assign a score out of 10 ",
         input_variables= ['essay'],
         partial_variables={'format_instruction' : parser.get_format_instructions()}
    
    )

    fomatted_prompt = prompt.format(essay = state['essay'])

    result = model.invoke(fomatted_prompt)
   
    output1 = parser.parse(result.content)
   

    return {
        'language_feedback' : output1.feedback,
        'individual_score' : [output1.score]
    }

In [83]:
def evaluation_analysis(state : EssayState):
    prompt = PromptTemplate(
         template = "Evaluate the depth of analysis of the followimng essay and provide a feedback and assign a score out of 10 ",
         input_variables= ['essay'],
         partial_variables={'format_instruction' : parser.get_format_instructions()}
    
    )

    fomatted_prompt = prompt.format(essay = state['essay'])


    result = model.invoke(fomatted_prompt)

    output2 = parser.parse(result.content)
   
    
   

    return {
        'analysis_feedback' : output2.feedback,
        'individual_score' : [output2.score] }

In [84]:
def evalution_thought(state : EssayState):
    prompt = PromptTemplate(
         template = "Evaluate the clarity of thought of the followimng essay and provide a feedback and assign a score out of 10 ",
         input_variables= ['essay'],
         partial_variables={'format_instruction' : parser.get_format_instructions()}
    
    )
    fomatted_prompt = prompt.format(essay = state['essay'])

    result = model.invoke(fomatted_prompt)

    output3 = parser.parse(result.content)
   
   
    
    return {
        'clarity_feedback' : output3.feedback,
        'individual_score' : [output3.score]
    }

In [85]:
def final_evaluation(state : EssayState):
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall = model.invoke(prompt).content

    #avg_calculate

    avg_score = sum(state['individual_score']) / len(state['individual_score'])

    return {
        'final_avg_score' : avg_score,
        'overall_feedback' : overall.feedback
    }


In [86]:
graph = StateGraph(EssayState)

graph.add_node('essay' , generate_essay)

graph.add_node('evaluation_lang' , evaluation_language)
graph.add_node('evaluation_anal' , evaluation_analysis)
graph.add_node('evaluation_thought' , evalution_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START , 'essay')

graph.add_edge('essay',"evaluation_lang")
graph.add_edge('essay',"evaluation_anal")
graph.add_edge('essay',"evaluation_thought")

graph.add_edge('evaluation_lang','final_evaluation')
graph.add_edge('evaluation_anal','final_evaluation')
graph.add_edge('evaluation_thought','final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()





In [ ]:
initial_state = {"topic": "The role of Artificial Intelligence in Indian Governance"}
result = workflow.invoke(initial_state)
print(result)

OutputParserException: Invalid json output: However, you didn't provide the essay. Please share the essay, and I'll evaluate the clarity of thought, provide feedback, and assign a score out of 10. 

Once you provide the essay, I'll be happy to assist you. 

(Note: The score will be based on the following criteria:

1. Organization and structure
2. Coherence and logical flow
3. Clarity and concision of language
4. Depth and accuracy of analysis
5. Use of evidence and examples)

Please paste the essay, and I'll provide a detailed feedback.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 